In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
# load all environment variables from .env
load_dotenv()
# Load groq API key into environment variable
os.environ["GROK_API_KEY"] = os.getenv("GROK_API_KEY")
## Load HuggingFace
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
# LangSmith Tracking configuration
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY_CONVERSATIONAL_QA"] = os.getenv("LANGCHAIN_API_KEY_CONVERSATIONAL_QA")
os.environ["LANGCHAIN_TRACKING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
groq_api_key = os.getenv("GROK_API_KEY")
huggingfacehub_api_token = os.getenv("HF_TOKEN")
llm_model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
llm_model

e:\Development\00-Repos\MyRepo\GenAiApps\LangChainvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002197FAC0D10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002197FC1F810>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [13]:
## Load HuggingFace Embeddings
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
import bs4
# Format retrieved docs (very common helper)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
store = {}  # In production, use Redis or database
## Session-based Chat History

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [15]:
# 1. Load, Chunk and index the contents of the blog to create a retriever

## Load the web page with only relevant sections (Data Ingestion)
loader = WebBaseLoader(
    web_path = ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content", "post-title", "post-header")
        )
    ),
)  
documents = loader.load()    
## Chunk the documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
text_splits = text_splitter.split_documents(documents) 
## Create the vector store
vector_store = Chroma.from_documents(documents = text_splits, embedding=embedding_model)
## Create the retriever
retriever = vector_store.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000219196091D0>, search_kwargs={})

In [20]:
## Create prompt template for Conversational QA
system_prompt = (
    "You are a helpful AI assistant that helps people find information "
    "about LangChain from a provided context. "
    "Use the context to answer the question at the end. "
    "If you don't know the answer, just say you don't know. "
    "Use three sentences to answer the question. "
    "Keep the answer concise and to the point."
    "\n\n"
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

## Create the Conversational QA chain (LCEL)
rag_chain = (
    {
        "context": retriever | format_docs,  # context = retriever → (retriever will be formated as string using format_docs)
        "input": RunnablePassthrough()       # keep the question as is (will be passed when invoking the chain)
    }
    | prompt                                 # fill in the prompt template
    | llm_model                              # llm model
    | StrOutputParser()                      # get clean string from the llm output
)
## Test the RAG chain
response = rag_chain.invoke("What is Self-Reflection?")
print(response)

Self-Reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable. In the context of LangChain, Self-Reflection is created by showing two-shot examples to LLM and each example is a pair of (failed trajectory, ideal reflection for guiding future changes in the plan).


In [34]:
## Create the retriever
retriever = vector_store.as_retriever()
## Create system message for question contextualization
contextualize_q_system_message = (
    " Given a chat history and the latest user question, "
    "which might reference context in the chat history,"
    "formulate a stanalone question which can be understood."
    "without the chat history. Do not answer the question,"
    "just reformulate it if needed otherwise return it as is."
)
## Create the Conversational QA chain (LCEL) - FIXED VERSION
rag_chain = (
    RunnablePassthrough.assign(
        context = lambda input_dict : format_docs(retriever.invoke(input_dict["input"])),
        # Pass chat_history through (it will be used by the prompt)
        chat_history = lambda input_dict : input_dict.get("chat_history", [])
    )
    | prompt  # prompt already has MessagesPlaceholder("chat_history")
    | llm_model
    | StrOutputParser()
)

# Wrap the chain with history
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key = "input",
    history_messages_key = "chat_history",
)

# 5. Use with session
question1 = "What is Self-Reflection?"
response1 = conversational_rag_chain.invoke(
    {"input": question1},
    config={"configurable": {"session_id": "user123"}}
)
print(response1)

## another way
# chat_history = []
# question = "What is Self-Reflection?"
# response1 = conversational_rag_chain.invoke(
#     {"input": question},
#     {"chat_history": chat_history}
# )
# chat_history.extend(
#     [
#         HumanMessage(content=question1),
#         AIMessage(content=response1)
#     ]
# )
# print(response1)


# Second question with history
question2 = "Explain more about it."
response2 = conversational_rag_chain.invoke(
    {"input": question2},
    config={"configurable": {"session_id": "user123"}}
)
print(response2)

Self-Reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable. Self-reflection is created by showing two-shot examples to a Large Language Model (LLM) and each example is a pair of (failed trajectory, ideal reflection for guiding future changes in the plan).
Unfortunately, the provided context does not provide enough information to give a detailed explanation of the topic. The context appears to be a combination of unrelated texts about the Reflexion framework, LSH (Locality-Sensitive Hashing), and ANNOY (Approximate Nearest Neighbors Oh Yeah). 

To provide a more detailed explanation, I would need more context or information about what you are referring to. However, I can provide a general overview of the three mentioned topics. 

Reflexion seems to be a framework related to artificial intelligence, possibly a rei